In [ ]:
!pip install -q pandas numpy scikit-learn nltk joblib imbalanced-learn

In [ ]:

from google.colab import drive
drive.mount('/content/drive')

import os, glob
PROJECT_ROOT = "/content/drive/MyDrive/radiological_report"
DATA_RAW_DIR = os.path.join(PROJECT_ROOT, "data", "raw")
DATA_PROC_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
os.makedirs(DATA_RAW_DIR, exist_ok=True)
os.makedirs(DATA_PROC_DIR, exist_ok=True)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW DATA FOLDER:", DATA_RAW_DIR)
print("Processed data will be saved to:", DATA_PROC_DIR)
print("Files in raw:", glob.glob(os.path.join(DATA_RAW_DIR,"*"))[:50])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROJECT_ROOT: /content/drive/MyDrive/radiological_report
RAW DATA FOLDER: /content/drive/MyDrive/radiological_report/data/raw
Processed data will be saved to: /content/drive/MyDrive/radiological_report/data/processed
Files in raw: ['/content/drive/MyDrive/radiological_report/data/raw/indiana_reports.csv', '/content/drive/MyDrive/radiological_report/data/raw/indiana_projections.csv']


In [ ]:

import pandas as pd, glob
candidates = glob.glob(os.path.join(DATA_RAW_DIR,"*.csv")) + glob.glob(os.path.join(DATA_RAW_DIR,"*.json"))
if not candidates:
    raise FileNotFoundError(f"No CSV/JSON files found in {DATA_RAW_DIR}. Upload your dataset files there first.")
DATA_FILE = candidates[0]
print("Using file:", DATA_FILE)

if DATA_FILE.lower().endswith(".csv"):
    df_sample = pd.read_csv(DATA_FILE, nrows=5000, low_memory=False)
else:
    df_sample = pd.read_json(DATA_FILE, lines=False, nrows=5000)
print("Columns:", df_sample.columns.tolist())
display(df_sample.head(3))

Using file: /content/drive/MyDrive/radiological_report/data/raw/indiana_reports.csv
Columns: ['uid', 'MeSH', 'Problems', 'image', 'indication', 'comparison', 'findings', 'impression']


,uid,MeSH,Problems,image,indication,comparison,findings,impression
0,1,normal,normal,Xray Chest PA and Lateral,Positive TB test,None.,The cardiac silhouette and mediastinum size ar...,Normal chest x-XXXX.
1,2,Cardiomegaly/borderline;Pulmonary Artery/enlarged,Cardiomegaly;Pulmonary Artery,"Chest, 2 views, frontal and lateral",Preop bariatric surgery.,None.,Borderline cardiomegaly. Midline sternotomy XX...,No acute pulmonary findings.
2,3,normal,normal,Xray Chest PA and Lateral,"rib pain after a XXXX, XXXX XXXX steps this XX...",NaN,NaN,"No displaced rib fractures, pneumothorax, or p..."


In [ ]:

candidates_cols = [c for c in df_sample.columns if any(k in c.lower() for k in ("report","finding","impression","text","description","note"))]
print("Auto-suggested columns for reports:", candidates_cols)
REPORT_COL = candidates_cols[0] if candidates_cols else input("Enter the report text column name: ")
print("REPORT_COL set to:", REPORT_COL)

Auto-suggested columns for reports: ['findings', 'impression']
REPORT_COL set to: findings


In [ ]:

import re
usecols = [REPORT_COL]
if DATA_FILE.lower().endswith(".csv"):
    df = pd.read_csv(DATA_FILE, usecols=usecols, low_memory=False)
else:
    df = pd.read_json(DATA_FILE, lines=False)
df = df.dropna(subset=[REPORT_COL]).reset_index(drop=True)
def normalize_text(s):
    s = str(s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s
df['report_clean'] = df[REPORT_COL].apply(normalize_text).str.lower()
print("Loaded rows:", len(df))
display(df[['report_clean']].head(3))

Loaded rows: 3337


,report_clean
0,the cardiac silhouette and mediastinum size ar...
1,borderline cardiomegaly. midline sternotomy xx...
2,there are diffuse bilateral interstitial and a...


In [ ]:

import random
negative_phrases = [
    "no acute", "no acute cardiopulmonary", "no acute cardiopulmonary disease",
    "no acute cardiopulmonary process", "clear lungs", "lungs are clear",
    "no acute abnormality", "no acute disease", "no focal consolidation",
    "no pneumothorax", "no pleural effusion", "normal chest", "no acute osseous abnormality"
]
def assign_label(text):
    for p in negative_phrases:
        if p in text:
            return 0
    if "normal" in text:
        return 0
    return 1
df['label'] = df['report_clean'].apply(assign_label)
print("Auto-label distribution:")
print(df['label'].value_counts())
print("\nExample Normal (auto):")
display(df[df['label']==0].sample(n=min(5, df[df['label']==0].shape[0]), random_state=42))
print("\nExample Abnormal (auto):")
display(df[df['label']==1].sample(n=min(5, df[df['label']==1].shape[0]), random_state=42))

Auto-label distribution:
label
0    3190
1     147
Name: count, dtype: int64

Example Normal (auto):


,findings,report_clean,label
1076,Heart is mildly enlarged stable. Mediastinal c...,heart is mildly enlarged stable. mediastinal c...,0
1046,Cardiac and mediastinal contours are unremarka...,cardiac and mediastinal contours are unremarka...,0
817,"The heart, pulmonary XXXX and mediastinum are ...","the heart, pulmonary xxxx and mediastinum are ...",0
428,There are no focal areas of consolidation. No ...,there are no focal areas of consolidation. no ...,0
1155,Heart size normal. Prominent epicardial fat. L...,heart size normal. prominent epicardial fat. l...,0



Example Abnormal (auto):


,findings,report_clean,label
2721,1. Cardiomegaly and/or pericardial effusion. 2...,1. cardiomegaly and/or pericardial effusion. 2...,1
1262,There are low lung volumes. The lungs are othe...,there are low lung volumes. the lungs are othe...,1
3174,There are midline sternotomy XXXX and mediasti...,there are midline sternotomy xxxx and mediasti...,1
492,The XXXX examination consists of frontal and l...,the xxxx examination consists of frontal and l...,1
2215,There is redemonstration of an AICD in the lef...,there is redemonstration of an aicd in the lef...,1


In [ ]:

from sklearn.model_selection import train_test_split
proc_path = os.path.join(DATA_PROC_DIR, "processed_reports.csv")
df[['report_clean','label']].to_csv(proc_path, index=False)
print("Saved processed CSV to:", proc_path)

train_df, temp_df = train_test_split(df[['report_clean','label']], test_size=0.3, random_state=42, stratify=df['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])
train_df.to_csv(os.path.join(DATA_PROC_DIR, "train.csv"), index=False)
val_df.to_csv(os.path.join(DATA_PROC_DIR, "val.csv"), index=False)
test_df.to_csv(os.path.join(DATA_PROC_DIR, "test.csv"), index=False)
print("Saved train/val/test to:", DATA_PROC_DIR)
print("Sizes -> train:", len(train_df), "val:", len(val_df), "test:", len(test_df))

Saved processed CSV to: /content/drive/MyDrive/radiological_report/data/processed/processed_reports.csv
Saved train/val/test to: /content/drive/MyDrive/radiological_report/data/processed
Sizes -> train: 2335 val: 501 test: 501
